# Gemma 4 12B Unified — exp5 ASR arm

**要做的事:執行階段 → 全部執行。就這樣。**
跑完最後一格會直接印出 Gemma 4 / Breeze / Gemini / Speechmatics 的對照表。

---

**為什麼要跑這個:** kikemu 的第二跳(譯)已經量過 Gemma 4 能打平 Gemini
(`results/report.md` §3F),但**第一跳(聽)在任何 API 上都拿不到**——
AI Studio 與 OpenRouter 只服務 `26B-A4B` 與 `31B`,兩者送音訊都回
`400 Audio input modality is not enabled for this model`。
會聽的是 HF 上 tag `any-to-any` 的 `E2B` / `E4B` / `12B-it`,只能自己跑。

**語料用 exp5**(台灣科技訪談 podcast 3 段 × M0/M3),因為 Breeze、
Gemini 批次、Gemini Live、Speechmatics 四個 arm 都在**同一批檔案**上跑過,
這是唯一能直接四方對照的地方。

> ⚠️ **一個要先知道的偏差**:exp5 的參考轉寫由 `gemini-3.5-flash` 產生,
> 所以 Gemini 系 arm 有主場優勢。Gemma 4 與 Breeze、SM 一樣都是外人,
> **Gemma vs Breeze、Gemma vs SM 的比較不受影響**;
> Gemma vs Gemini 要記得這條(報告 §3E.7)。


## 1. GPU 檢查與載入策略

`gemma-4-12B-it` 是 bf16 **23.9 GB** 權重。這格會依 GPU 記憶體自動選:

| GPU | 策略 |
|---|---|
| ≥ 34 GB(A100 40G / H100) | **bf16 原生**——與模型卡一致,最乾淨 |
| 20~34 GB(L4 24G / A10) | 4-bit NF4 量化 |
| < 20 GB(T4 16G) | 4-bit NF4;若仍 OOM,把 `MODEL_ID` 換成 `google/gemma-4-E4B-it` |

**量化不是原生設定**,所以會寫進每筆結果的 meta,報告裡要照實標。


In [ ]:
import importlib, importlib.util, subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or '(無輸出)')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('沒有 GPU。執行階段 → 變更執行階段類型 → GPU,再重跑。')

# 🚨 只裝缺的,而且把 numpy 釘死。
# Colab 的科學堆疊(numpy / scipy / numba / librosa / torch / torchvision)是互相
# 配對好的,任何一個被 pip 動到就會連鎖爆炸。這裡踩過兩次:
#   · `-U torchvision` → RuntimeError: operator torchvision::nms does not exist
#   · `-U librosa`     → 連帶升 numba → 動到 numpy →
#                        ImportError: cannot import name '_center' from numpy._core.umath
# 兩次都只能「執行階段 → 中斷連線並刪除執行階段」重來,restart 救不回。
# 所以:librosa / soundfile / torchvision Colab 都內建,一律不要碰。
import numpy
NUMPY_PIN = numpy.__version__

def _ge(mod, minver):
    """→ 這個模組是否存在且版本 >= minver"""
    if importlib.util.find_spec(mod) is None:
        return False
    try:
        v = importlib.import_module(mod).__version__
        return tuple(int(x) for x in v.split('.')[:len(minver)]) >= minver
    except Exception:
        return False

pkgs = []
if not _ge('transformers', (5, 15)):                 pkgs.append('transformers>=5.15')
if importlib.util.find_spec('accelerate') is None:   pkgs.append('accelerate')
if importlib.util.find_spec('bitsandbytes') is None: pkgs.append('bitsandbytes')
if pkgs:
    print('要安裝:', pkgs, f'(numpy 釘在 {NUMPY_PIN})')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    *pkgs, f'numpy=={NUMPY_PIN}'], check=True)
else:
    print('相依套件都已具備,不動 pip')

# 安裝後把整條鏈 import 一次:壞了現在就要知道,不要跑了半小時才炸。
for m in ('numpy', 'scipy', 'librosa', 'soundfile', 'torch', 'torchvision', 'transformers'):
    importlib.invalidate_caches()
    try:
        importlib.import_module(m)
    except Exception as e:
        raise RuntimeError(
            f'{m} 壞了({type(e).__name__}: {e})\n'
            'pip 動到了 Colab 內建的科學堆疊。\n'
            '修法:執行階段 → 中斷連線並刪除執行階段 → 重新全部執行。') from e

import transformers, torchvision
print('numpy', numpy.__version__, '| torchvision', torchvision.__version__, '(內建,未動)')

import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

GB = torch.cuda.get_device_properties(0).total_memory / 1e9
LOAD_4BIT = GB < 34
CAP = torch.cuda.get_device_capability(0)

print(f'GPU {torch.cuda.get_device_name(0)}  {GB:.0f} GB  compute {CAP[0]}.{CAP[1]}  →  '
      f"{'4-bit NF4 量化' if LOAD_4BIT else 'bf16 原生'}")
# CHUNK_SEC 的自動調整在下一格(參數格)——那裡才定義得到它。
print('transformers', transformers.__version__, '| torch', torch.__version__)




## 2. 參數


In [ ]:
REPO     = 'clarencechien/kikemu'
BRANCH   = 'claude/improve-experiment-credibility-c8ekb2'

# ── 模型 ─────────────────────────────────────────────────────────────
# 三個有音訊的 Gemma 4 變體(HF tag = any-to-any):
#   google/gemma-4-12B-it   23.9 GB 權重 — 要 A100/L4,T4 跑不動(見下)
#   google/gemma-4-E4B-it   較小,T4 可行
#   google/gemma-4-E2B-it   最小
# (26B-A4B 與 31B **沒有音訊**,送音訊回 400,不要選。)
MODEL_ID = 'google/gemma-4-12B-it'
ARM      = 'Xgma_12b'          # 換模型時記得一起改,不然結果會蓋掉別的 arm
CONDS    = ['M0', 'M3']        # 與其他 arm 一致:只取兩端

# ── 為什麼 T4 跑不動 12B ────────────────────────────────────────────
# 5 分鐘音訊 ≈ 7500 個 audio token。T4(compute 7.5)吃不到 SDPA 的
# flash / mem-efficient backend,會退回 math backend 把注意力矩陣實體化:
#   7500² × 16 heads × 4 bytes ≈ 3.4 GB   ← 正好是它 OOM 要不到的那一塊
# 權重塞得進去(4-bit 約 7 GB),注意力塞不進去。
#
# CHUNK_SEC:把音訊切成幾秒一段分別轉寫再接起來。0 = 整段送(預設)。
# 下一格會在偵測到 T4 級記憶體時自動設成 60。
# ⚠️ 切塊是**有代價的**:模型看不到跨塊的上下文,對專名不利,
#    而且其他 arm 都是整段處理。切塊跑出來的數字會寫進 meta,
#    報告裡必須標明,不能和整段的數字混在一起比。
CHUNK_SEC = 0

# T4(<20GB / compute 7.5)整段送 5 分鐘會 OOM 在注意力那一步,自動切塊。
# (GB 由上一格算出。)
if GB < 20 and CHUNK_SEC == 0:
    CHUNK_SEC = 60
    print('⚠️ 記憶體不足以整段處理 5 分鐘音訊,自動改為 60 秒切塊。')
    print('   切塊會讓模型看不到跨塊上下文,對專名不利,而其他 arm 都是整段處理。')
    print('   這個設定會寫進結果 meta,報告裡必須標明,不可與整段數字混比。')
    print('   要拿乾淨可比的數字,請改用 L4 / A100,或改跑 google/gemma-4-E4B-it。')
print(f'切塊設定:{f"{CHUNK_SEC} 秒" if CHUNK_SEC else "整段送"}')

# 與 exp5 的 Gbat arm **逐字相同**的樸素 prompt。唯一變數要是模型,不是 prompt。
PROMPT = ('逐字轉寫這段音訊。說話者混用中文與英文,英文詞請原樣保留,不要翻譯。'
          '不要摘要、不要加說話者標記、不要加時間碼,只輸出轉寫文字。')

# chat template 的 enable_thinking 預設就是 False,關閉時會送出一個立刻閉合的
# <|channel>thought\n<channel|>。這等同於 API 端的 thinkingLevel:'minimal',
# 也就是 CLAUDE.md 鐵律 4 要求的設定。不要改成 True。
ENABLE_THINKING = False



## 3. 取語料 → 加噪 → 對指紋

全部由 repo 裡的腳本完成,notebook 不自己實作。最後對
`exp5/corpus/audio_manifest.json` 的 SHA256——指紋相符就代表你手上的音檔
與其他四個 arm 跑的是同一份。**不需要雲端硬碟**,音檔來自公開 RSS。


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

WORK = Path('/content/kikemu')
if WORK.exists():
    # 已經 clone 過就拉最新的——不然修正推上去了,你這邊還是舊版。
    subprocess.run(['git','-C',str(WORK),'fetch','--depth','1','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(WORK),'reset','--hard',f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,
                    f'https://github.com/{REPO}.git', str(WORK)], check=True)
os.chdir(WORK)

def sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()

for script in ('exp5/scripts/prep_audio.py','exp5/scripts/degrade.py'):
    r = subprocess.run([sys.executable, script], cwd=WORK, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr.strip()); raise RuntimeError(f'{script} 失敗,原因見上方 stderr')

MAN   = json.loads((WORK/'exp5/corpus/audio_manifest.json').read_text())
picks = json.loads((WORK/'exp5/corpus/picks.json').read_text())
targets = [f"{p['seg']}__{c}" for p in picks for c in CONDS]
print('\n聲學條件指紋:')
for stem in targets:
    ok = sha256(WORK/'exp5/corpus/conditions'/f'{stem}.wav') == MAN['conditions'][stem]['sha256']
    print(f"  {stem}  {'✅ bit-identical' if ok else '⚠️ 不符'}")
print(f'\n本次 {len(targets)} 個檔')



## 4. 載入模型


In [ ]:
import time, torch
from transformers import AutoProcessor, Gemma4UnifiedForConditionalGeneration

# T4 / V100 是 compute 7.x,**沒有 bf16 tensor core**——指定 bfloat16 會走軟體模擬,
# 慢到不切實際(實測 T4 + 12B + 60s 切塊約 17~25 分鐘/chunk)。
# Ampere(8.0)以上才有 bf16。所以這裡依 compute capability 選:
#   >= 8.0 → bfloat16(模型卡的原生設定)
#   <  8.0 → float16 (吃得到 T4 的 fp16 tensor core)
# ⚠️ Gemma 家族在 fp16 下有數值溢位(NaN)的前科。第 6 格的健全性檢查會抓到
#    空輸出/亂碼——**那一格沒過就不要信結果**,老實換 L4 / A100 跑 bf16。
DTYPE = torch.bfloat16 if CAP >= (8, 0) else torch.float16
print(f'compute {CAP[0]}.{CAP[1]} → dtype {str(DTYPE).split(chr(46))[-1]}')

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
kw = {'dtype': DTYPE, 'device_map': 'cuda'}
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    # 音訊塔與投影層**不要量化**:Gemma4UnifiedMultimodalEmbedder.forward 只在
    # `weight.dtype.is_floating_point` 為真時才把 float32 的 input_features 轉型;
    # 量化後 weight 是 uint8,那個轉型不會發生。它們只佔幾百 MB。
    kw['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=['embed_audio', 'embed_vision', 'lm_head'])
model = Gemma4UnifiedForConditionalGeneration.from_pretrained(MODEL_ID, **kw).eval()
print(f'載入完成 {time.time()-t0:.0f}s | '
      f"{'4bit(音訊塔保留 ' + str(DTYPE).split(chr(46))[-1] + ')' if LOAD_4BIT else str(DTYPE)}")
print('audio embedder dtype:', model.model.embed_audio.embedding_projection.weight.dtype)


## 5. 推論

輸出欄位對齊既有 arm(`score.py` 讀 `transcript`,檔名 `<seg>__<cond>.json`),
每筆記下音檔 sha256 與載入策略,結果 JSON 自己就能證明跑的是哪一份、怎麼跑的。

**解碼要濾掉 thought channel**:Gemma 4 會用 `<channel|>` 分隔思考與答案,
不濾就會把推理文字當成轉寫存進去(這個坑在 API 端已經踩過一次,
見 `docs/gemini-api-lessons.md`)。


In [ ]:
import json, time, librosa, numpy as np

RAW = WORK/'exp5/results/raw'/ARM
RAW.mkdir(parents=True, exist_ok=True)
meta_env = {'model': MODEL_ID, 'arm': ARM, 'api': 'local transformers',
            'load': '4bit-nf4' if LOAD_4BIT else 'bf16',
            'chunk_sec': CHUNK_SEC, 'whole_file': CHUNK_SEC == 0,
            'enable_thinking': ENABLE_THINKING, 'do_sample': False,
            'prompt': PROMPT, 'transformers': transformers.__version__,
            'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
            'corpus': 'exp5'}
(RAW/'_meta.json').write_text(json.dumps(meta_env, ensure_ascii=False, indent=1))

def strip_thought(text: str) -> str:
    """Gemma 4 用 <channel|> 閉合 thought channel;答案是最後一段。"""
    return text.split('<channel|>')[-1].strip()

def transcribe(audio):
    msgs = [{'role':'user','content':[{'type':'audio','audio':audio},
                                      {'type':'text','text':PROMPT}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
        enable_thinking=ENABLE_THINKING).to(model.device)
    n_in = inputs['input_ids'].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=4096, do_sample=False)
    txt = processor.decode(out[0][n_in:], skip_special_tokens=True)
    del inputs, out
    torch.cuda.empty_cache()
    return strip_thought(txt), n_in

for stem in targets:
    dst = RAW/f'{stem}.json'
    if dst.exists():
        print(f'  跳過(已有){stem}'); continue
    wav = WORK/'exp5/corpus/conditions'/f'{stem}.wav'
    audio, _ = librosa.load(wav, sr=16000, mono=True)
    t0 = time.time()
    if CHUNK_SEC:
        step = CHUNK_SEC * 16000
        pieces, n_tok = [], 0
        for i in range(0, len(audio), step):
            seg_audio = audio[i:i+step]
            if len(seg_audio) < 16000:      # 不足 1 秒的尾巴丟掉
                continue
            txt, n = transcribe(seg_audio)
            pieces.append(txt); n_tok += n
            print(f'    {stem} chunk {i//step+1}: {len(txt)} 字', flush=True)
        text = ''.join(pieces)
    else:
        text, n_tok = transcribe(audio)
    el = time.time() - t0
    digest = sha256(wav)
    dst.write_text(json.dumps({
        'arm': ARM, 'file': f'{stem}.wav',
        'audio_s': round(len(audio)/16000, 1),
        'transcript': text,
        'meta': {**meta_env, 'elapsed_sec': round(el,1), 'audio_sha256': digest,
                 'audio_matches_manifest': digest == MAN['conditions'][stem]['sha256'],
                 'prompt_tokens_total': int(n_tok)},
    }, ensure_ascii=False, indent=1))
    print(f'  {stem}  {el:.0f}s  {len(text)} 字'
          f"{f'(切 {CHUNK_SEC}s)' if CHUNK_SEC else ''}", flush=True)


## 6. 健全性檢查

跑分之前先確認輸出**不是**下列三種壞掉的形狀。這一格失敗就不要信下面的數字。


In [ ]:
import re
bad = []
for stem in targets:
    d = json.loads((RAW/f'{stem}.json').read_text())
    t = d['transcript']
    cjk = sum(1 for c in t if '\u4e00' <= c <= '\u9fff')
    ratio = cjk/max(len(t),1)
    # ① 空的或極短 ② 中文字比例過低(答錯語言 / 吐出英文推理)③ 重複迴圈
    rep = max((len(m.group(0)) for m in re.finditer(r'(.{4,20}?)\1{3,}', t)), default=0)
    flag = []
    if len(t) < 200: flag.append('過短')
    if ratio < 0.3: flag.append(f'中文比例僅 {ratio:.2f}')
    if rep > 80:    flag.append(f'重複迴圈 {rep} 字')
    print(f"  {stem}  {len(t):>5} 字  中文 {ratio:.2f}  {'⚠️ '+'/'.join(flag) if flag else '✅'}")
    if flag: bad.append(stem)
print('\n' + ('全部通過 ✅' if not bad else f'⚠️ 有問題:{bad} — 先看原文再決定要不要採信'))
print('\n樣本(T1__M0 前 300 字):')
print(json.loads((RAW/'T1__M0.json').read_text())['transcript'][:300])


## 7. 對照表:Gemma 4 vs Breeze vs Gemini vs Speechmatics

用 repo 裡既有的 `score.py` / `compare.py`,**判定規則與其他 arm 完全相同**
(strict / tolerant 兩套、簡繁折疊、術語變體表都是凍結的)。


In [ ]:
r = subprocess.run([sys.executable,'exp5/scripts/score.py'], cwd=WORK, capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
r = subprocess.run([sys.executable,'exp5/scripts/compare.py'], cwd=WORK, capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())


## 8. 帶走

把結果打包下載,交回給 repo 就能寫進報告。


In [ ]:
import shutil
z = shutil.make_archive('/content/xgma-exp5','zip', WORK/'exp5/results/raw', ARM)
print('已打包:', z)
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print('自動下載失敗,請從左側檔案面板下載:', e)
